# MozzareLLM: gene cluster analysis

MozzareLLM analyzes gene clusters from functional genomics screens with a large language model. For each cluster it:

1. identifies the biological pathway(s) that best explain why the genes cluster together (or states that no coherent pathway exists),
2. categorizes every gene relative to that pathway — ESTABLISHED (documented role in the pathway), NOVEL_ROLE (documented elsewhere; membership here is the new evidence), or UNCHARACTERIZED (annotation too sparse to judge) — with an evidence-ladder subclass for the latter two,
3. prioritizes the understudied genes for experimental follow-up.

**How it works.** Your input is a cluster table (one row per gene, with a cluster assignment) and a short screen-context JSON describing the assay. The pipeline maps genes to stable UniProt accessions, builds one evidence bundle per cluster (functional annotations, optionally your per-gene phenotypic features), and sends each bundle to the model with a structured prompt. Every run writes:

| Output | Contents |
|---|---|
| `traces/cluster_<id>.json` | full per-cluster record: raw response, tool calls, tokens, cost |
| `<screen>_clusters.json` | parsed per-cluster results (the raw structured output) |
| `<screen>_genes.csv` | one row per gene: category, subclass, rationale, evidence, pathway call |
| `<screen>_clusters.csv` | one row per cluster: pathway, confidence, per-category counts, coverage |

The CSVs are the tables to read; the JSONs are the complete record.


## Setup

Requires an `ANTHROPIC_API_KEY` in a `.env` file at the repo root (see `README.md`).


In [ ]:
import os
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from mozzarellm.clients.llm_api_clients import create_client
from mozzarellm.pipeline.screen_analysis import analyze_screen, prepare_screen_bundles

load_dotenv()
client = create_client(model="claude-sonnet-5", api_key=os.getenv("ANTHROPIC_API_KEY"))

EXAMPLES = Path("..") / "examples"
OUTPUT = Path("output")
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")


## Analyzing your own screen

Two inputs:

1. **Cluster table** — CSV/TSV/XLSX with a gene column and a cluster column (defaults: `gene_symbol`, `cluster`). Optional per-gene phenotypic feature columns can be included in the analysis (Example 1).
2. **Screen context** — copy `screen_context_template.json`, fill in your assay, perturbation, readout, and clustering details. This grounds the model in what your clusters mean.

Then run `prepare_screen_bundles` and `analyze_screen` exactly as in the examples below, pointing at your files. Bundles are cached: re-running skips the annotation lookups.


## Example 1 — Optical pooled screen, with phenotypic features

Funk et al. 2022 morphological clusters. This example includes each gene's **phenotypic features** (`up_features` / `down_features` — the imaging features driven up or down by knockout) in the evidence bundles, and the prompt adds feature-interpretation reasoning steps: the model must check whether its pathway call is consistent with the observed morphology, not just the literature.


In [ ]:
ops_bundles = prepare_screen_bundles(
    screen_name="funk_2022",
    cluster_table=EXAMPLES / "ops" / "funk_2022.csv",
    output_dir=OUTPUT,
    feature_columns=["up_features", "down_features"],
)
ops = analyze_screen(
    screen_name="funk_2022",
    cluster_to_bundle_map=ops_bundles,
    client=client,
    run_dir=OUTPUT / "funk_2022_analysis" / f"run_{STAMP}_cot_feat",
    screen_context_path=EXAMPLES / "ops" / "screen_context.json",
    mode="cot",
    include_features=True,
)
print(f"cost: ${ops['total_cost_usd']}  errors: {ops['errors']}")
ops["cluster_df"]


In [ ]:
ops["gene_df"].head(15)


## Example 2 — DepMap co-essentiality, baseline analysis

Wainberg et al. 2021 co-essentiality modules. No feature columns here — this is the default path: a single chain-of-thought call per cluster over the annotation evidence alone. Use this shape for any clustering without per-gene phenotype measurements.


In [ ]:
depmap_bundles = prepare_screen_bundles(
    screen_name="wainberg_2021",
    cluster_table=EXAMPLES / "depmap" / "wainberg_2021.csv",
    output_dir=OUTPUT,
)
depmap = analyze_screen(
    screen_name="wainberg_2021",
    cluster_to_bundle_map=depmap_bundles,
    client=client,
    run_dir=OUTPUT / "wainberg_2021_analysis" / f"run_{STAMP}_cot",
    screen_context_path=EXAMPLES / "depmap" / "screen_context.json",
    mode="cot",
)
depmap["cluster_df"]


## Example 3 — Proteomics co-abundance, with literature validation

Schaffer et al. 2025 protein co-abundance clusters, with **PubMed literature validation** (`mcp=True`): the model is given search tools and a validation step that checks its NOVEL_ROLE and UNCHARACTERIZED calls against retrieved literature before finalizing. Slower and costlier per cluster; use when the flagged-gene calls will drive experiments.


In [ ]:
prot_bundles = prepare_screen_bundles(
    screen_name="schaffer_2025",
    cluster_table=EXAMPLES / "proteomics" / "schaffer_2025.csv",
    output_dir=OUTPUT,
)
prot = analyze_screen(
    screen_name="schaffer_2025",
    cluster_to_bundle_map=prot_bundles,
    client=client,
    run_dir=OUTPUT / "schaffer_2025_analysis" / f"run_{STAMP}_cot_mcp",
    screen_context_path=EXAMPLES / "proteomics" / "screen_context.json",
    mode="cot",
    mcp=True,
)
prot["cluster_df"]


## Reading the results

- Start with `<screen>_clusters.csv`: the pathway call and confidence per cluster, and how completely the cluster was classified (`classification_completeness`, `missed_genes`).
- Then `<screen>_genes.csv`, filtered to `category != "ESTABLISHED"`: these are the follow-up candidates, each with its evidence subclass and the model's rationale.
- A cluster with `dominant_process = "No coherent biological pathway"` and empty gene lists is a deliberate abstention, not a failure.
- The per-cluster `traces/` JSONs hold the complete record (raw response, literature tool calls, tokens, cost) for any call you want to audit.
